# Regresi Tarif Harian Hotel (ADR) - Hotel Booking Demand

Notebook ini memprediksi **ADR (Average Daily Rate / tarif harian)** pemesanan hotel, yaitu nilai numerik kontinu, sehingga masalahnya adalah **regresi**.

Lima algoritma dibandingkan: **Linear Regression, Ridge, Lasso, Random Forest, dan Gradient Boosting**. Metrik evaluasi: **MAE, MSE, RMSE, dan R²**.

Setiap cell kode diikuti penjelasan singkat.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('hotel_bookings.csv')
print("Ukuran Dataset (Baris, Kolom):", df.shape)
display(df[['adr']].describe().T)

- `pd.read_csv` membaca dataset ke DataFrame `df`.
- `df[['adr']].describe()` menampilkan ringkasan statistik target: rata-rata, standar deviasi, minimum, kuartil, dan maksimum.

Terlihat nilai `adr` minimum negatif dan maksimum sangat besar (5400). Keduanya adalah outlier/data tidak wajar yang perlu dibersihkan.

Tujuan cell: Memuat data dan mengenali sebaran target regresi.

In [ ]:
# Membersihkan data target (ADR)
# Tarif harus positif (adr <= 0 adalah data tidak wajar / kamar gratis) dan dibatasi maksimum 500 untuk membuang outlier ekstrem
df = df[(df['adr'] > 0) & (df['adr'] <= 500)].copy()

# Mengisi nilai kosong
df['children'] = df['children'].fillna(0)
df['country'] = df['country'].fillna('Unknown')

print("Ukuran data setelah dibersihkan:", df.shape)

plt.figure(figsize=(7, 4))
sns.histplot(df['adr'], bins=50, kde=True)
plt.title('Sebaran ADR (Tarif Harian)')
plt.xlabel('ADR')
plt.tight_layout()
plt.show()

- `df[(df['adr'] > 0) & (df['adr'] <= 500)]` hanya menyimpan baris dengan tarif positif dan tidak lebih dari 500.
- `.copy()` membuat salinan agar tidak muncul peringatan saat data diubah.
- `fillna` mengisi nilai kosong `children` dengan 0 dan `country` dengan 'Unknown'.
- `sns.histplot` membuat histogram untuk melihat sebaran tarif.

Batas 500 adalah keputusan analisis untuk membuang outlier ekstrem, bukan aturan baku. Batas ini bisa kamu ubah dan jelaskan di laporan.

Tujuan cell: Membersihkan target dan fitur dari data tidak wajar.

In [ ]:
# Menentukan fitur (X) dan target (y)
# Kolom kategorikal diubah menjadi kolom 0/1 dengan One-Hot Encoding (get_dummies)
num_features = ['lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights',
                'adults', 'children', 'babies', 'is_repeated_guest',
                'previous_cancellations', 'booking_changes',
                'required_car_parking_spaces', 'total_of_special_requests',
                'arrival_date_week_number']
cat_features = ['hotel', 'arrival_date_month', 'meal', 'market_segment',
                'deposit_type', 'customer_type']

X = pd.get_dummies(df[num_features + cat_features], columns=cat_features, drop_first=True)
y = df['adr']

print("Dimensi X:", X.shape)
print("Dimensi y:", y.shape)

- `num_features` adalah kolom angka, `cat_features` adalah kolom teks/kategori.
- `pd.get_dummies(...)` mengubah tiap kategori menjadi kolom 0/1 (One-Hot Encoding). Ini lebih tepat untuk regresi linear dibanding Label Encoding, karena Label Encoding membuat urutan angka palsu (misalnya 'Mei' dianggap lebih besar dari 'April').
- `drop_first=True` membuang satu kategori per kolom agar tidak redundan.
- `y = df['adr']` adalah target yang diprediksi.

Kolom `is_canceled` dan `reservation_status` sengaja tidak dipakai karena tidak relevan untuk memprediksi tarif.

Tujuan cell: Menyiapkan data numerik penuh yang bisa diproses semua algoritma.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print("Data Latih:", len(X_train), "| Data Uji:", len(X_test))

- `train_test_split` membagi data 80% latih dan 20% uji.
- `random_state=42` membuat pembagian selalu sama setiap dijalankan.

Tujuan cell: Memisahkan data untuk belajar dan data untuk menguji model.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Model linear memakai StandardScaler agar semua fitur berskala sama (penting untuk Ridge dan Lasso)
models = {
    'Linear Regression': make_pipeline(StandardScaler(), LinearRegression()),
    'Ridge (L2)': make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    'Lasso (L1)': make_pipeline(StandardScaler(), Lasso(alpha=0.01)),
    'Random Forest': RandomForestRegressor(n_estimators=100, min_samples_leaf=3,
                                           n_jobs=-1, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=4,
                                                   learning_rate=0.1, random_state=42),
}

hasil = []
prediksi = {}
for nama, m in models.items():
    m.fit(X_train, y_train)
    pred_train = m.predict(X_train)
    pred = m.predict(X_test)
    prediksi[nama] = pred
    mse = mean_squared_error(y_test, pred)
    hasil.append({
        'Model': nama,
        'MAE': mean_absolute_error(y_test, pred),
        'MSE': mse,
        'RMSE': np.sqrt(mse),
        'R2 Uji': r2_score(y_test, pred),
        'R2 Latih': r2_score(y_train, pred_train),
    })

hasil = pd.DataFrame(hasil).set_index('Model').round(3)
display(hasil.sort_values('R2 Uji', ascending=False))

- `StandardScaler` menyamakan skala fitur. Ini diperlukan Ridge dan Lasso karena penaltinya sensitif terhadap skala.
- `make_pipeline` menggabungkan scaler dan model dalam satu rangkaian, sehingga scaler hanya belajar dari data latih.
- **Linear Regression**: baseline paling sederhana.
- **Ridge (L2)**: regresi linear dengan penalti kuadrat koefisien, baik untuk fitur yang saling berkorelasi.
- **Lasso (L1)**: penalti nilai absolut yang bisa membuat koefisien fitur kurang penting menjadi 0 (seleksi fitur otomatis).
- **Random Forest**: kumpulan banyak pohon keputusan, mampu menangkap hubungan non-linear.
- **Gradient Boosting**: pohon dibangun berurutan, tiap pohon memperbaiki kesalahan pohon sebelumnya.
- `alpha` mengatur kekuatan penalti. Semakin besar, semakin kuat regularisasi.

Cara membaca metrik:
- **MAE**: rata-rata selisih absolut prediksi dan nilai asli, satuannya sama dengan ADR. Semakin kecil semakin baik.
- **MSE**: rata-rata selisih kuadrat. **RMSE** adalah akar MSE sehingga satuannya kembali sama dengan ADR.
- **R²**: maksimal 1. Mendekati 1 berarti model menjelaskan variasi data dengan baik, mendekati 0 berarti tidak lebih baik dari menebak rata-rata, dan bisa negatif jika lebih buruk dari itu.
- Jika **R² Latih jauh lebih besar dari R² Uji**, model overfitting.

Tujuan cell: Melatih lima algoritma regresi dan membandingkan performanya dengan metrik yang sama.

In [ ]:
# Visualisasi perbandingan R2 dan MAE antar model
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
hasil['R2 Uji'].sort_values().plot.barh(ax=ax[0], color='steelblue')
ax[0].set_title('R² Data Uji (makin besar makin baik)')
hasil['MAE'].sort_values(ascending=False).plot.barh(ax=ax[1], color='darkorange')
ax[1].set_title('MAE Data Uji (makin kecil makin baik)')
plt.tight_layout()
plt.show()

# Aktual vs Prediksi untuk model terbaik
terbaik = hasil['R2 Uji'].idxmax()
print("Model terbaik berdasarkan R2 Uji:", terbaik)
plt.figure(figsize=(5, 5))
plt.scatter(y_test, prediksi[terbaik], s=3, alpha=0.3)
plt.plot([0, 500], [0, 500], 'r--')
plt.xlabel('ADR Aktual'); plt.ylabel('ADR Prediksi')
plt.title(f'Aktual vs Prediksi - {terbaik}')
plt.tight_layout()
plt.show()

- Grafik batang kiri membandingkan R², grafik kanan membandingkan MAE.
- `idxmax()` mencari model dengan R² uji tertinggi.
- Scatter plot membandingkan nilai aktual dengan prediksi. Semakin dekat titik ke garis merah putus-putus, semakin akurat prediksinya.

Tujuan cell: Memvisualisasikan perbandingan model dan ketepatan model terbaik.

In [ ]:
# Koefisien Lasso: fitur yang koefisiennya 0 dianggap tidak berpengaruh
lasso = models['Lasso (L1)'].named_steps['lasso']
koef = pd.Series(lasso.coef_, index=X.columns)
print("Fitur yang dibuang Lasso (koefisien = 0):", int((koef == 0).sum()), "dari", len(koef))
display(koef[koef != 0].sort_values(key=abs, ascending=False).head(10).round(3))

# Fitur terpenting menurut Gradient Boosting
imp = pd.Series(models['Gradient Boosting'].feature_importances_, index=X.columns)
imp.sort_values().tail(10).plot.barh(figsize=(7, 4), title='10 Fitur Terpenting - Gradient Boosting')
plt.tight_layout()
plt.show()

- `lasso.coef_` berisi koefisien tiap fitur. Koefisien 0 berarti fitur dibuang Lasso.
- `feature_importances_` pada Gradient Boosting menunjukkan seberapa besar kontribusi tiap fitur.

Bagian ini membantu menjelaskan *mengapa* suatu algoritma cocok atau tidak, misalnya Lasso untuk seleksi fitur dan Gradient Boosting untuk pola non-linear.

Tujuan cell: Menafsirkan model, bukan hanya melihat angka akurasi.

## Kesimpulan

1. **Model terbaik adalah Random Forest**, dengan R² data uji 0,815, MAE 11,65, dan RMSE 20,20. Artinya rata-rata prediksi ADR meleset sekitar 11-12 satuan tarif, dan model menjelaskan sekitar 81,5% variasi tarif. Gradient Boosting berada di urutan kedua (R² 0,743; MAE 16,71).
2. **Model linear (Linear Regression, Ridge, dan Lasso) hasilnya hampir identik** (R² sekitar 0,522; MAE sekitar 23,93). Selisih R² yang besar terhadap model berbasis pohon (±0,3) menunjukkan bahwa hubungan antara fitur dan ADR bersifat **non-linear** (misalnya pengaruh musim, jenis hotel, dan segmen pasar yang saling berinteraksi), sehingga model linear kurang cocok untuk data ini.
3. **Regularisasi (Ridge/Lasso) tidak memberi peningkatan** karena jumlah fitur sedikit (40 kolom) dibanding data yang besar (±117 ribu baris), sehingga risiko overfitting pada model linear kecil. Dengan `alpha=0.01`, Lasso belum menghapus satu fitur pun.
4. **Random Forest cenderung overfitting ringan** (R² latih 0,917 vs R² uji 0,815), sedangkan Gradient Boosting lebih stabil (0,753 vs 0,743) tetapi masih bisa ditingkatkan lewat tuning `n_estimators`, `max_depth`, dan `learning_rate`.
5. **Kesesuaian dengan literatur:** studi-studi terbaru juga menemukan bahwa model berbasis pohon dan ensemble mengungguli model linear pada data tabular, misalnya untuk prediksi harga kamar hotel (Oliveira et al., 2024) dan prediksi regresi umum (Quayesam, 2024; Aljuboori, 2024). Pada studi-studi tersebut Gradient Boosting sering menjadi yang terbaik, sedangkan pada notebook ini Random Forest unggul. Perbedaan ini wajar karena Gradient Boosting di sini belum di-tuning dan karakteristik datanya berbeda.
6. **Pemilihan algoritma:** untuk data ini yang non-linear dan berukuran menengah-besar, Random Forest atau Gradient Boosting lebih sesuai. Linear Regression, Ridge, dan Lasso tetap berguna sebagai baseline dan bila interpretasi koefisien dibutuhkan (domain sensitif).

**Keterbatasan:** data dengan ADR ≤ 0 atau > 500 dibuang sebagai outlier; tidak ada fitur lokasi atau harga pesaing; hasil dapat sedikit berbeda bila versi pustaka berbeda.

## Referensi

Antonio, N., de Almeida, A., & Nunes, L. (2019). Hotel booking demand datasets. *Data in Brief, 22*, 41-49. https://doi.org/10.1016/j.dib.2018.11.126

Oliveira, F. B., Silva-Filho, M. W., Barbosa, G. A., Freitas, J. P., Penna, C., & Miranda, P. B. C. (2024). Machine learning and time series analysis to forecast hotel room prices. *Brazilian Conference on Intelligent Systems (BRACIS 2024)*, Belém/PA. Sociedade Brasileira de Computação. https://sol.sbc.org.br/index.php/bracis/article/view/33638

Binesh, F., & Belarmino, A. M. (2024). Forecasting hotel room prices when entering turbulent times: A game-theoretic artificial neural network model. *International Journal of Contemporary Hospitality Management*. https://doi.org/10.1108/IJCHM-10-2022-1233

Quayesam, D. L. (2024). *A comparison of machine learning algorithms for predicting sea surface temperature in the Great Barrier Reef region*. arXiv:2411.15202. https://arxiv.org/abs/2411.15202

Aljuboori, A. A. (2024). Enhancing accuracy in predicting continuous values through regression. *University of Bahrain Scientific Journals*. https://journals.uob.edu.bh/handle/123456789/5847

Mao, M. (2024). A comparative study of random forest regression for predicting house prices using gradient boosting. In *Proceedings of the 2023 International Conference on Data Science, Advanced Algorithm and Intelligent Computing (DAI 2023)*. Atlantis Press. https://doi.org/10.2991/978-94-6463-370-2_63

Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825-2830.